# 示例 3：拟合尖峰响应

以下示例演示了矢量拟合功能，该示例使用了从 scikit-rf 的 `tests` 文件夹中复制的 4 端口示例网络。由于该网络在各个响应中存在许多谐振点，因此拟合起来有些复杂。可以在 [矢量拟合教程](../../tutorials/VectorFitting.ipynb) 中找到更多说明和背景信息。

In [ ]:
import matplotlib.pyplot as mplt
import numpy as np

import skrf
from skrf.vectorFitting import VectorFitting

要创建一个 `VectorFitting` 实例，需要传递一个包含 N 端口频率响应的 `Network` 对象。在本示例中，使用了 `skrf/tests` 文件夹中的 `Agilent_E5071B.s4p` 文件的副本：

In [ ]:
nw = skrf.network.Network('./Agilent_E5071B.s4p')
vf = VectorFitting(nw)

为了进行比较，我们将使用常规的矢量拟合例程 `vector_fit()` 以及自动矢量拟合例程 `auto_fit()` 来拟合这个网络。

## 常规矢量拟合法

对于常规的矢量拟合法，需要指定初始极点的数量和类型，这取决于响应的*行为*。作为一个经验法则，可以统计各个响应中谐振或“峰”的数量，作为初始猜测值。在这种情况下，4端口网络有16个需要拟合的响应。如图中幅值图所示，$S_{11}$ 和一些其他响应都非常*尖锐*，每个响应大约有15个局部最大值，并且局部最大值之间有大约相同数量的局部最小值。其他响应只有5-6个局部最大值，或者它们非常嘈杂，幅值非常小（例如$S_{24}$ 和 $S_{42}$）。假设 $S_{11}$ 的大多数 15 个最大值都出现在与其他响应的最大值相同的频率上，那么可以预期需要 15 个共轭极点来进行拟合。由于这可能并不完全正确，因此尝试使用 20-30 个极点应该是一个很好的起点，可以拟合所有响应中的所有谐振。

In [ ]:
# plot magnitudes of all 16 responses in the 4-port network
fig, ax = mplt.subplots(4, 4)
fig.set_size_inches(12, 8)
for i in range(4):
    for j in range(4):
        nw.plot_s_mag(i, j, ax=ax[i][j])
        ax[i][j].get_legend().remove()
fig.tight_layout()
mplt.show()

在尝试了不同数量的实数极点和共轭极点后，发现以下设置可以得到非常好的拟合结果。初始模型阶数为 $1 + 2 * 26 = 53$，在拟合过程中将保持不变。还发现其他设置也适用于该网络（例如，0-2 个实数极点和 25-26 个共轭极点）：

In [ ]:
vf.vector_fit(n_poles_real=1, n_poles_cmplx=26)

In [ ]:
print(f'model order = {vf.get_model_order(vf.poles)}')

收敛性也可以通过收敛曲线进行检查：

In [ ]:
vf.plot_convergence()

拟合的模型参数现在存储在类属性 `poles`、`residues`、`proportional_coeff` 和 `constant_coeff` 中，以便后续使用。为了验证结果，可以将拟合的模型响应与原始网络响应进行比较。由于该模型会在任何给定的频率处返回响应，因此有必要检查其在原始样本频率范围之外的响应。在本例中，原始网络的测量范围为 0.5 GHz 到 4.5 GHz，因此我们可以绘制从直流到 10 GHz 的拟合曲线：

In [ ]:
freqs = np.linspace(0, 10e9, 501)
fig, ax = mplt.subplots(4, 4)
fig.set_size_inches(12, 8)
for i in range(4):
    for j in range(4):
        vf.plot_s_mag(i, j, freqs=freqs, ax=ax[i][j])
        ax[i][j].get_legend().remove()
fig.tight_layout()
mplt.show()

In [ ]:
print(f'rms error = {vf.get_rms_error()}')

如图所示，获得了非常好的拟合结果。这也可以从低于 0.01 的低均方根误差中看出。但是，打印了一条关于非被动拟合的 UserWarning（参见上面 `vector_fit()` 的输出）。对模型被动性的评估和强制执行在 [此示例](./vectorfitting_ex4_passivity.ipynb) 中有更详细的描述，这对于电路模拟器中的某些用例（例如瞬态模拟）非常重要。为了在电路模拟中使用该模型，可以基于拟合参数创建等效电路。目前，这仅针对 SPICE 进行了实现，但等效电路的结构可以应用于任何类型的电路模拟器。

`vf.write_spice_subcircuit_s('/home/vinc/Desktop/4-port_model.sp')`

导出的 `.sp` 文件随后可以作为子电路导入到 SPICE 中。请参阅 [环形槽示例](./vectorfitting_ex1_ringslot.ipynb)。

## 自动矢量拟合

对于以下自动矢量拟合例程，用户无需指定起始极点。默认设置应该足以实现成功的拟合，因为所需的极点数量会在过程中进行优化。不过，更改 `auto_fit()` 的参数可以进一步提高收敛性并降低模型阶数。

In [ ]:
vf.auto_fit()

In [ ]:
print(f'model order = {vf.get_model_order(vf.poles)}')
print(f'n_poles_real = {np.sum(vf.poles.imag == 0.0)}')
print(f'n_poles_complex = {np.sum(vf.poles.imag > 0.0)}')

In [ ]:
print(f'rms error = {vf.get_rms_error()}')

In [ ]:
freqs = np.linspace(0, 10e9, 501)
fig, ax = mplt.subplots(4, 4)
fig.set_size_inches(12, 8)
for i in range(4):
    for j in range(4):
        vf.plot_s_mag(i, j, freqs=freqs, ax=ax[i][j])
        ax[i][j].get_legend().remove()
fig.tight_layout()
mplt.show()